Libraries and Setup

In [53]:
import numpy as np
import pandas as pd
import joblib
import os
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_percentage_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import warnings

warnings.filterwarnings('ignore')

print("Libraries imported and ready for Optimized Training")

Libraries imported and ready for Optimized Training


Data Loading

In [54]:
X_train_scaled = np.load('models/X_train_scaled.npy')
y_train_scaled = np.load('models/y_train_scaled.npy')
X_test_scaled = np.load('models/X_test_scaled.npy')
y_test_scaled = np.load('models/y_test_scaled.npy')

scaler_y = joblib.load('models/scaler_y.pkl')

print("Data loaded successfully")
print(f"Training set: {X_train_scaled.shape}")
print(f"Testing set: {X_test_scaled.shape}")
print(f"Total Features: {X_train_scaled.shape[1]}")

Data loaded successfully
Training set: (1464, 12)
Testing set: (366, 12)
Total Features: 12


Linear Regression Training

In [ ]:
from sklearn.svm import SVR
lr_model = SVR(kernel='rbf', C=10.0, epsilon=0.01, gamma='scale') 

lr_model.fit(X_train_scaled, y_train_scaled.ravel())
score_lr = evaluate_real_world("Support Vector Regressor (Base)", lr_model, X_test_scaled, y_test_scaled)

Training Support Vector Regressor (Replacing failing Linear Model)...
SVR training complete.
Support Vector Regressor (Base) Performance
   R2 Score : 0.2750
   MAPE     : 67.00%
   RMSE     : 3601.72



Random Forest Training

In [56]:
rf_model = RandomForestRegressor(n_estimators=500, max_depth=15, random_state=42, n_jobs=-1)
rf_model.fit(X_train_scaled, y_train_scaled.ravel())

print("Random Forest training complete")

Random Forest training complete


XGBoost Training

In [57]:
xgb_model = xgb.XGBRegressor(
    n_estimators=500, 
    learning_rate=0.03, 
    max_depth=10, 
    objective='reg:squarederror',
    subsample=0.8,
    colsample_bytree=0.8
)
xgb_model.fit(X_train_scaled, y_train_scaled.ravel())

print("XGBoost training complete")

XGBoost training complete


Advanced Evaluation Function

In [58]:
def evaluate_real_world(name, model, X_t, y_t_scaled):

    preds_scaled = model.predict(X_t).reshape(-1, 1)

    preds_log = scaler_y.inverse_transform(preds_scaled)
    y_true_log = scaler_y.inverse_transform(y_t_scaled)

    preds_actual = np.expm1(preds_log)
    y_true_actual = np.expm1(y_true_log)

    r2 = r2_score(y_true_actual, preds_actual)
    rmse = np.sqrt(mean_squared_error(y_true_actual, preds_actual))
    mape = mean_absolute_percentage_error(y_true_actual, preds_actual) * 100
    
    print(f"{name} Performance")
    print(f"   R2 Score : {r2:.4f}")
    print(f"   MAPE     : {mape:.2f}%")
    print(f"   RMSE     : {rmse:.2f}\n")
    
    return r2, mape

print("Advanced Evaluation function ready")

Advanced Evaluation function ready


Performance Results

In [59]:
score_lr = evaluate_real_world("Linear Regression", lr_model, X_test_scaled, y_test_scaled)
score_rf = evaluate_real_world("Random Forest", rf_model, X_test_scaled, y_test_scaled)
score_xgb = evaluate_real_world("XGBoost", xgb_model, X_test_scaled, y_test_scaled)

Linear Regression Performance
   R2 Score : 0.2750
   MAPE     : 67.00%
   RMSE     : 3601.72

Random Forest Performance
   R2 Score : 0.8785
   MAPE     : 15.36%
   RMSE     : 1474.53

XGBoost Performance
   R2 Score : 0.8961
   MAPE     : 14.02%
   RMSE     : 1363.32



Save Models

In [60]:
joblib.dump(lr_model, 'models/linear_regression_model.pkl')
joblib.dump(rf_model, 'models/random_forest_model.pkl')
joblib.dump(xgb_model, 'models/xgboost_model.pkl')

print("All models saved successfully")

All models saved successfully


Statistical Summary

In [61]:
import matplotlib.pyplot as plt

importances = xgb_model.feature_importances_

print("Feature Importance Summary:")
for i, v in enumerate(importances):
    print(f"Feature {i}: {v:.4f}")

print(f"\nFinal Comparison (MAPE):")
print(f"Linear Regression : {score_lr[1]:.2f}%")
print(f"Random Forest     : {score_rf[1]:.2f}%")
print(f"XGBoost           : {score_xgb[1]:.2f}%")

Feature Importance Summary:
Feature 0: 0.0019
Feature 1: 0.0019
Feature 2: 0.0018
Feature 3: 0.0059
Feature 4: 0.0081
Feature 5: 0.0085
Feature 6: 0.0050
Feature 7: 0.0072
Feature 8: 0.0416
Feature 9: 0.5156
Feature 10: 0.3996
Feature 11: 0.0028

Final Comparison (MAPE):
Linear Regression : 67.00%
Random Forest     : 15.36%
XGBoost           : 14.02%
